# 실험 S 추가: form 말투를 빼고 학습 (S_minus_form, 사후 선택, 논문에 없는 추가)
설계·예측은 `EXPERIMENT_DESIGN.md` 4.4절 "추가 실험 S_minus_form" (돌리기 전에 적어 둠).

노트북 12 결과: ours 가 크게 깨진 말투는 **form** 하나였다 (`topic: ...; question: ...` 같은 key: value 꼴).
ours 의 form negative 거절 82.4% vs S_all 99.5%. 학습에서 뺀 howto·fragment 는 ours 도 이미 잘해서 판정을 못 했다.
그래서 **form 을 빼고** 나머지 4개 말투로 넓게 학습한 모델이 form 에서 어디에 가까운지 본다. 결과를 보고 고른 말투라 "사후 선택" 이라고 밝힌다.

| 모델 | 학습 말투 | 이 노트북에서 |
|---|---|---|
| ours, S_all, S_minus_howto, S_minus_fragment | (노트북 12) | 채점 결과 재사용 |
| **S_minus_form** | orig + howto·fragment·need·nonnative | 학습 + 채점 |

판정 (form 에서): 메운 비율 = (ours − S_minus_form) / (ours − S_all). **0.5 이상 → 일반화, 미만 → 말투 암기**.
주 지표 negative 거절률, 보조 positive 엄격 정확도.

**사용자 예측** (돌리기 전에): S_minus_form 의 form negative 거절률은 ours(82.4%) 와 S_all(99.5%) 중 어디에 가까울까? 정확도(69.6% vs 84.8%)는?
> (여기에 예측)

런타임: **A100 GPU + 고용량 RAM (80GB)**. 🔑 Secrets 에 `HF_TOKEN`. 학습 1번 + 채점 1번, 30분 정도

## 준비 (그냥 실행)

In [ ]:
# 다른 노트북과 같은 버전 (최신 5.x 는 API 가 바뀌어 에러가 난다)
!pip -q install transformers==4.56.2 accelerate
from huggingface_hub import login
from google.colab import userdata, drive

login(userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

import os
REPO = "/content/octopus_v2_reproduce"
if not os.path.exists(REPO):
    !git clone https://github.com/nahcub/octopus_v2_reproduce.git {REPO}
%cd {REPO}
!git pull
!git log --oneline -1

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers, os, json, datetime; print("transformers", transformers.__version__)  # 4.56.2 여야 함

DRIVE = "/content/drive/MyDrive/octopus_v2"
EVAL = f"{DRIVE}/runs/eval/style"
TEST = "data/style/dev_styled.jsonl"
N_TEST = sum(1 for _ in open(TEST))

RUNS = ["S_all", "S_minus_howto", "S_minus_fragment", "S_minus_form"]   # 학습 파일 data/style/train_<이름>.jsonl
MODELS = {"ours": f"{DRIVE}/runs/full", **{n: f"{DRIVE}/runs/{n}" for n in RUNS}}
DELETE_AFTER_EVAL = False              # True 면 채점 끝난 모델 가중치를 Drive 에서 지운다 (probs 는 남김)

def done_training(name): return name == "ours" or os.path.exists(f"{MODELS[name]}/train_args.json")
def done_eval(name): return os.path.exists(f"{EVAL}/{name}/probs.jsonl") and sum(1 for _ in open(f"{EVAL}/{name}/probs.jsonl")) == N_TEST
print("채점 문항", N_TEST, {n: ("채점 끝" if done_eval(n) else "학습 끝" if done_training(n) else "아직") for n in MODELS})
# ours·A00 과 같은 조건(fp32 가중치 + AdamW, batch 16)은 A100 80GB 에서만 들어간다
import torch
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
assert GPU_GB > 60, f"GPU {GPU_GB:.0f} GiB: A100 80GB 가 필요하다. 런타임 연결 해제 후 A100 + 고용량 RAM 으로 다시 연결"
print(f"GPU {GPU_GB:.0f} GiB OK")

---
## 1. 학습 데이터 확인
form 말투가 0개인지, positive·negative 가 나머지 말투를 비슷하게 나눠 갖는지

In [ ]:
import collections
rows = [json.loads(l) for l in open("data/style/train_S_minus_form.jsonl")]
c = collections.Counter((r["kind"][:3], r["style"]) for r in rows)
print(len(rows), dict(sorted(c.items())))
assert not any(r["style"] == "form" for r in rows)

---
## 2. 학습 → 말투별 test 채점
이미 끝난 모델(노트북 12의 4개)은 건너뛰고 S_minus_form 만 학습·채점한다

In [ ]:
for name, path in MODELS.items():
    print(f"\n==================== {name}")
    if not done_training(name):
        print("학습 시작", datetime.datetime.now())
        !python -m src.train --train-file data/style/train_{name}.jsonl --output-dir {path}
        print("학습 끝", datetime.datetime.now())
    if not done_eval(name):
        !python -m src.probs generate --model "{path}" --test-file {TEST} --out {EVAL}/{name}/probs.jsonl
    if DELETE_AFTER_EVAL and name != "ours" and done_eval(name):
        !rm -rf "{path}"/*.safetensors

---
## 3. 말투별 거절 — 판정
`S_minus_form` 의 form 칸 (`*`) 과 맨 아래 `S_minus_form 말투 form` 줄을 본다. 노트북 12 결과(`style_report.txt`)는 그대로 두고 `style_report_form.txt` 로 따로 저장

In [ ]:
args = " ".join(f"--model {n} {EVAL}/{n}/probs.jsonl" for n in MODELS if done_eval(n))
!python -m src.style_report --test-file {TEST} {args} --out {EVAL}/style_report_form.json | tee {EVAL}/style_report_form.txt

---
## 끝나면
결과: Drive `runs/eval/style/style_report_form.txt`·`.json`, `runs/eval/style/S_minus_form/probs.jsonl`.
랩탑에서 Claude 에게 "노트북 13 분석해줘" 라고 하면 설계 문서 4.4절 예측과 비교한다.